# WordPiece
BERT(Bidirectional Encoder Representations from Transformers):

保留常见的基础词汇（如 "play"），而将复杂的生僻词或派生词拆分成有意义的子词（如 "unbelievable" 拆分成 "un", "##believ", "##able"）。这样既控制了词表大小，又解决了 OOV 问题，同时保留了充足的语义信息。

在查看 BERT 的分词结果时，你经常会看到带有 ## 前缀的词（例如 ##ing）。

* 没有 ## 的词：表示这是一个完整词汇的开头。
* 带有 ## 的词：表示这个子词依附于它前面的词，是前面词汇的延续。

## 训练

WordPiece 和另一种子词算法 BPE（Byte-Pair Encoding）非常相似，都是自底向上合并的贪心算法，但它们的合并标准不同。BPE 是单纯合并出现频率最高的相邻子词对；而 WordPiece 是合并能让训练数据似然度提升最大的子词对。具体步骤如下：

1. 初始化： 将训练语料库中的所有词汇拆分成单个字符，并加入特殊的控制符（如 [CLS], [SEP], [UNK]），形成初始词表。此时除了单词首字母外，其余字母都加上 `##` 前缀。
2. 建立语言模型： 基于当前的词表对语料库建立一元语言模型（Unigram Language Model）。
3. 计算得分： 遍历语料库中所有相邻的子词对（例如子词 $A$ 和子词 $B$），计算合并它们带来的收益。WordPiece 的评分核心实际上是计算两个子词的互信息（PMI）：
   $$Score = \frac{P(AB)}{P(A) \cdot P(B)}$$
   其中 $P(AB)$ 是 $A$ 和 $B$ 组合在一起出现的概率，$P(A)$ 和 $P(B)$ 是它们单独出现的概率。这个公式的直观意义是：如果两个子词经常一起出现，且很少单独出现（即 $P(AB)$ 很大，而 $P(A)$ 和 $P(B)$ 很小），那么合并它们的得分就极高。
4. 合并： 挑选得分最高的子词对，将其合并成一个新的子词（如 $AB$），并加入词表中。
5. 循环： 重复步骤 2 到 4，直到词表达到预设的大小（例如 BERT 的词表大小为 30,522），或者合并操作带来的收益低于某个阈值。

## 推理

当模型在实际应用中遇到一段新文本时, WordPiece 使用的是最长匹配原则（Maximum Forward Matching）。假设我们遇到一个词：`unaffable`:

1. 算法会从左到右扫描，寻找词表中能匹配到的最长的前缀。假设它在词表中找到了 "un"。
2. 接着处理剩下的部分 "affable"。由于这不是词的开头，算法会寻找带有 ## 的最长匹配串。假设它找到了 "##aff"。
3. 继续处理剩下的 "able"。算法在词表中找到了 "##able"。
4. 最终，"unaffable" 被切分为 ["un", "##aff", "##able"]。

如果算法在任何一步找不到任何可以匹配的子词（即使退化到单个字符也找不到），整个原始单词就会被无情地标记为 [UNK]（在英文中极为罕见，但在处理多语言或特殊符号时可能发生）。

## 实现

### 第一步：准备数据与基础构建

In [ ]:
import re  # 导入正则库，用来清洗文本中的符号
import torch  # 导入 PyTorch，后续会把 token id 转成张量
from collections import defaultdict  # 导入带默认值的字典，便于计数
from datasets import load_dataset  # 导入 Hugging Face 数据集加载函数

# 1. 使用 Hugging Face 非流式数据集（一次性加载 split）
dataset = load_dataset(  # 调用数据集加载函数
    "wikitext",  # 指定数据集名称
    "wikitext-2-raw-v1",  # 指定该数据集的具体配置
    split="train",  # 只加载训练集 split
)  # 结束 load_dataset 调用

# 2. 从完整数据集中抽样，构造小语料
max_samples = 500  # 设定最多取 500 条有效文本，降低后续训练开销
corpus = []  # 创建一个列表，用来保存清洗后的句子

for text in dataset["text"]:  # 遍历训练集中的 text 字段
    text = (text or "").strip()  # 防止空值并去掉首尾空格
    if not text:  # 如果这一条是空文本
        continue  # 跳过当前样本，处理下一条
    text = re.sub(r"[^a-zA-Z\s]", " ", text).lower()  # 仅保留英文字母和空格，再转小写
    text = re.sub(r"\s+", " ", text).strip()  # 把多个连续空格压缩成一个，并去掉首尾空格
    if text:  # 只有清洗后仍非空才保留
        corpus.append(text)  # 把有效文本加入语料列表
    if len(corpus) >= max_samples:  # 如果已达到采样上限
        break  # 提前结束循环

# 3. 预分词（Pre-tokenization）：按空格切分单词，并统计词频
word_freqs = defaultdict(int)  # 创建词频字典，默认计数为 0
for text in corpus:  # 遍历语料中的每一句文本
    words = text.split()  # 按空格切成单词列表
    for word in words:  # 遍历句子里的每个单词
        word_freqs[word] += 1  # 对该单词的出现次数加 1

print(f"非流式采样完成，句子数: {len(corpus)}")  # 打印最终采样得到的句子数量
print(f"原始 train split 样本数: {len(dataset)}")  # 打印训练集总样本数
print("基础词频统计(前20项):", dict(list(word_freqs.items())[:20]))  # 打印前 20 个词频示例

非流式采样完成，句子数: 500
原始 train split 样本数: 36718
基础词频统计(前20项): {'valkyria': 54, 'chronicles': 39, 'iii': 17, 'senj': 5, 'no': 29, 'unrecorded': 1, 'japanese': 6, 'lit': 4, 'of': 1120, 'the': 2646, 'battlefield': 8, 'commonly': 2, 'referred': 4, 'to': 775, 'as': 283, 'outside': 13, 'japan': 4, 'is': 141, 'a': 680, 'tactical': 3}


### 第二步：初始化字符级词表与拆分

WordPiece 的起点是将所有单词打碎成单个字母。除了首字母，其他字母都要加上 ## 前缀。

In [ ]:
# 初始化词表（包含基础字符）
vocab = set()  # 用集合存 token，自动去重
# 记录每个单词当前的拆分状态
splits = {}  # 用字典记录每个单词被拆成了哪些子词

for word in word_freqs.keys():  # 遍历所有出现过的单词
    # 首字母不加 ##，后续字母加 ##
    split = [word[0]] + ["##" + c for c in word[1:]]  # 把单词拆成 WordPiece 初始形式
    splits[word] = split  # 保存该单词当前的拆分结果
    vocab.update(split)  # 把这些子词加入词表集合

# 添加特殊 Token
special_tokens = ["<PAD>", "<UNK>", "<CLS>", "<SEP>"]  # 定义常见特殊标记
vocab.update(special_tokens)  # 把特殊标记也加入词表

print("初始词表大小:", len(vocab))  # 打印初始化后的词表大小
print("单词 'learning' 的初始拆分:", splits['learning'])  # 查看示例单词的拆分结果

初始词表大小: 56
单词 'learning' 的初始拆分: ['l', '##e', '##a', '##r', '##n', '##i', '##n', '##g']


### 第三步：定义核心评分与合并逻辑
WordPiece 的核心是挑选得分最高的相邻子词对进行合并。我们使用的评分公式为：

$$Score = \frac{freq(A, B)}{freq(A) \cdot freq(B)}$$

注意：现实中为了防止分母过大导致长词无法合并，通常会结合频次阈值（在前面实现过），这里我们实现其最核心的互信息思想。

In [ ]:
def compute_pair_scores(splits, word_freqs):  # 定义函数：计算每个相邻子词对的 WordPiece 分数
    """计算相邻子词对的得分"""  # 函数文档字符串
    pair_freqs = defaultdict(int)  # 记录每个相邻子词对出现的频率
    token_freqs = defaultdict(int)  # 记录每个子词单独出现的频率
    
    # 统计独立 token 频率和相邻 pair 频率
    for word, split in splits.items():  # 遍历每个单词及其当前拆分
        freq = word_freqs[word]  # 取出该单词在语料中的词频
        for i in range(len(split)):  # 遍历该单词拆分后的每个子词位置
            token_freqs[split[i]] += freq  # 子词频率按单词频率累计
            if i < len(split) - 1:  # 如果不是最后一个子词，则可以组成相邻对
                pair_freqs[(split[i], split[i+1])] += freq  # 统计相邻子词对频率
                
    # 计算 WordPiece 得分
    pair_scores = {}  # 创建字典保存每个相邻对的得分
    for pair, freq in pair_freqs.items():  # 遍历每个相邻对子及其频率
        score = freq / (token_freqs[pair[0]] * token_freqs[pair[1]])  # 按公式计算得分
        pair_scores[pair] = score  # 保存该相邻对的得分
        
    return pair_scores  # 返回所有相邻对的得分

def merge_pair(a, b, splits):  # 定义函数：把指定相邻对子词 (a, b) 合并
    """将拆分状态中的特定相邻对 (a, b) 合并为 ab"""  # 函数文档字符串
    for word, split in splits.items():  # 遍历每个单词及其拆分
        if len(split) == 1:  # 如果该单词当前只有一个子词
            continue  # 无法继续合并，直接跳过
        i = 0  # 从拆分列表的第 0 个位置开始扫描
        while i < len(split) - 1:  # 只要还存在相邻对就继续
            if split[i] == a and split[i+1] == b:  # 如果当前位置刚好匹配待合并对子
                # 合并操作
                # 注意处理 ## 逻辑：如果 b 有 ##，合并后去掉 b 的 ##
                merged_token = a + b[2:] if b.startswith("##") else a + b  # 生成合并后的新子词
                split = split[:i] + [merged_token] + split[i+2:]  # 用新子词替换原来的两个子词
            else:  # 如果当前位置不匹配待合并对子
                i += 1  # 指针向后移动一个位置
        splits[word] = split  # 更新该单词的最新拆分状态
    return splits  # 返回更新后的所有拆分状态

### 第四步：执行训练循环构建最终词表
我们设定一个目标词表大小，不断循环“打分 -> 寻找最高分 -> 合并”的过程。

In [ ]:
vocab_size = 1400  # 设定目标词表大小

while len(vocab) < vocab_size:  # 只要当前词表还没达到目标大小，就继续训练
    scores = compute_pair_scores(splits, word_freqs)  # 计算当前所有相邻对子词的得分
    if not scores:  # 如果已经没有可计算的相邻对子词
        break  # 结束训练循环
        
    # 找到得分最高的组合
    best_pair = max(scores, key=scores.get)  # 选出分数最高的相邻对子词
    
    # 执行合并
    splits = merge_pair(best_pair[0], best_pair[1], splits)  # 在所有单词拆分中执行这次最佳合并
    
    # 将新生成的 token 加入词表
    new_token = best_pair[0] + best_pair[1][2:] if best_pair[1].startswith("##") else best_pair[0] + best_pair[1]  # 生成新 token，并处理 ## 前缀
    vocab.add(new_token)  # 把新 token 放入词表

# 为了后续 PyTorch 使用，我们需要建立 Token 到 ID 的映射字典
token2id = {token: i for i, token in enumerate(sorted(vocab))}  # 按排序后的 token 生成稳定的 token->id 映射
print(f"训练完成！最终词表大小: {len(vocab)}")  # 打印最终词表大小
print("\n部分词表展示:")  # 打印提示信息
for token, tid in list(token2id.items())[:15]:  # 仅展示前 15 个 token-id 对
    print(f"{token}: {tid}")  # 输出 token 及其对应 id

训练完成！最终词表大小: 1400

部分词表展示:
##a: 0
##abcock: 1
##ack: 2
##acking: 3
##acqu: 4
##adnought: 5
##adow: 6
##aff: 7
##affi: 8
##affic: 9
##ainbows: 10
##ajor: 11
##aksmith: 12
##akthrough: 13
##amp: 14


### 第五步：实现推理阶段的分词器与 PyTorch 转换
模型训练好后，我们需要实现“最长前缀匹配”来对新句子进行分词，并将它们转化为 PyTorch 的 Tensor。

In [ ]:
def encode_wordpiece(text, token2id):  # 定义函数：把输入文本做 WordPiece 分词并转成张量
    """最长前缀匹配分词，并转换为 PyTorch Tensor"""  # 函数文档字符串
    words = text.lower().split()  # 先转小写，再按空格切分成词
    encoded_tokens = []  # 用于保存分词后的 token 序列
    
    for word in words:  # 逐个处理输入中的单词
        start = 0  # start 表示当前还未匹配部分的起点
        while start < len(word):  # 只要单词还有未匹配部分就继续
            end = len(word)  # 每次先从最长候选子串开始尝试
            best_token = "<UNK>"  # 默认本轮找不到匹配时用 <UNK>
            
            # 从最长可能寻找匹配
            while start < end:  # 逐步缩短候选子串长度进行匹配
                sub_str = word[start:end]  # 截取当前候选子串
                if start > 0:  # 如果不是词首位置
                    sub_str = "##" + sub_str  # 按 WordPiece 规则给后续子词加 ## 前缀
                    
                if sub_str in token2id:  # 如果候选子串在词表里
                    best_token = sub_str  # 记录这个可用 token
                    break  # 找到最长匹配后就结束内层循环
                end -= 1  # 若未匹配成功，则缩短候选子串继续尝试
                
            if best_token == "<UNK>":  # 如果这一段完全找不到任何匹配
                # 如果找不到匹配，按照严格 WordPiece，整个词变为 UNK
                encoded_tokens = ["<UNK>"]  # 严格模式下把整个输入标记为未知
                break  # 停止处理当前句子
            else:  # 如果找到匹配子词
                encoded_tokens.append(best_token)  # 把匹配到的 token 加入结果
                start = end  # 起点前移到未匹配部分，继续处理
                
    # 转换为 ID
    input_ids = [token2id.get(token, token2id["<UNK>"]) for token in encoded_tokens]  # 把 token 序列映射成 id 序列
    
    # === PyTorch 核心转化 ===
    # 将普通的 Python list 转换为 PyTorch 可以进行梯度计算的 Tensor
    tensor_ids = torch.tensor(input_ids, dtype=torch.long)  # 把 id 列表转成 long 类型张量
    
    return encoded_tokens, tensor_ids  # 返回分词结果和对应张量

# 测试一下我们自己写的模型！
test_sentence = "machine unlearning" # unlearning 是未见过的组合，但词根存在
tokens, tensor_output = encode_wordpiece(test_sentence, token2id)  # 调用编码函数处理测试句子

print(f"测试句子: '{test_sentence}'")  # 打印原始测试句子
print(f"分词结果: {tokens}")  # 打印 WordPiece 分词结果
print(f"PyTorch Tensor 输出: {tensor_output}")  # 打印张量化后的 id 序列
print(f"Tensor 形状: {tensor_output.shape}")  # 打印张量形状

测试句子: 'machine unlearning'
分词结果: ['m', '##a', '##ch', '##i', '##n', '##e', 'un', '##l', '##e', '##a', '##r', '##n', '##ing']
PyTorch Tensor 输出: tensor([1007,    0,   45,   81,  169,   66, 1286,  126,   66,    0,  235,  169,
          97])
Tensor 形状: torch.Size([13])
